In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/combined_2019_2021.logs.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(['station_code','timestamp'])
df = df.reset_index(drop=True)

print("Loaded:", df.shape)

Loaded: (3056615, 5)


In [15]:
df['hour']         = df['timestamp'].dt.hour
df['day_of_week']  = df['timestamp'].dt.dayofweek
df['month']        = df['timestamp'].dt.month
df['year']         = df['timestamp'].dt.year
df['is_weekend']   = df['day_of_week'].isin([5,6]).astype(int)
df['is_peak_hour'] = df['hour'].isin(
    [8,9,17,18,19]).astype(int)

# Cyclical encoding
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin']  = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos']  = np.cos(2 * np.pi * df['day_of_week'] / 7)

print("Time features done")
print(df[['hour','is_weekend','is_peak_hour',
          'hour_sin','hour_cos']].head())

Time features done
   hour  is_weekend  is_peak_hour  hour_sin  hour_cos
0    20           0             0 -0.866025  0.500000
1    21           0             0 -0.707107  0.707107
2    22           0             0 -0.500000  0.866025
3    23           0             0 -0.258819  0.965926
4     0           0             0  0.000000  1.000000


In [ ]:
import sys
sys.path.append('../utils')
from india_events import get_event_info

df['date_str'] = df['timestamp'].dt.strftime('%Y-%m-%d')

event_rows = []
for date in df['date_str']:
    event_rows.append(get_event_info(date))

event_df = pd.DataFrame(event_rows)
df = pd.concat([
    df.reset_index(drop=True),
    event_df.reset_index(drop=True)
], axis=1)

print("India event features done")
print(df[['date_str','is_festival',
          'festival_name','overall_mult']].head(10))

India event features done
     date_str  is_festival festival_name  overall_mult
0  2018-12-31            0          None           1.0
1  2018-12-31            0          None           1.0
2  2018-12-31            0          None           1.0
3  2018-12-31            0          None           1.0
4  2019-01-01            0          None           1.0
5  2019-01-01            0          None           1.0
6  2019-01-01            0          None           1.0
7  2019-01-01            0          None           1.0
8  2019-01-01            0          None           1.0
9  2019-01-01            0          None           1.0


In [17]:
for lag in [1, 2, 3, 4, 8, 24]:
    df[f'lag_{lag}'] = df.groupby(
        'station_code')['people_in'].shift(lag)

print("Lag features done")
print(df[['people_in',
          'lag_1','lag_4','lag_24']].head(30).tail(5))

Lag features done
    people_in   lag_1   lag_4  lag_24
25       2211  2165.0   502.0   321.0
26       2689  2211.0  1764.0   348.0
27       3088  2689.0  3650.0   741.0
28       3108  3088.0  2165.0   940.0
29       2406  3108.0  2211.0  1401.0


In [18]:
for window in [4, 8, 24]:
    df[f'roll_mean_{window}'] = (
        df.groupby('station_code')['people_in']
        .transform(lambda x:
            x.shift(1).rolling(window, min_periods=1).mean())
    )
    df[f'roll_std_{window}'] = (
        df.groupby('station_code')['people_in']
        .transform(lambda x:
            x.shift(1).rolling(window, min_periods=1).std())
    )

print("Rolling features done")

Rolling features done


In [19]:
before = len(df)
df = df.dropna()
after  = len(df)

print(f"Rows before: {before:,}")
print(f"Rows after:  {after:,}")
print(f"Dropped:     {before-after:,}")

df.to_csv('../data/features.csv', index=False)
print("\nSaved to data/features.csv")
print("Final shape:", df.shape)
print("\nAll columns:")
for col in df.columns:
    print(f"  - {col}")

Rows before: 3,056,615
Rows after:  3,049,823
Dropped:     6,792

Saved to data/features.csv
Final shape: (3049823, 34)

All columns:
  - timestamp
  - station_code
  - people_in
  - people_out
  - year
  - hour
  - day_of_week
  - month
  - is_weekend
  - is_peak_hour
  - hour_sin
  - hour_cos
  - dow_sin
  - dow_cos
  - date_str
  - is_festival
  - festival_name
  - festival_mult
  - is_cricket
  - is_bandh
  - is_monsoon
  - overall_mult
  - lag_1
  - lag_2
  - lag_3
  - lag_4
  - lag_8
  - lag_24
  - roll_mean_4
  - roll_std_4
  - roll_mean_8
  - roll_std_8
  - roll_mean_24
  - roll_std_24
